# Notebook 3: ASV Inference and Denoising via DADA2

This notebook executes the core bioinformatics workflow for the **PRJEB36531** cohort (558 preterm infant samples). 
The primary objectives of this notebook are:
1. **Quality Profiling**: Inspecting the base-wise quality scores of the trimmed FASTQ files.
2. **Learning Error Rates**: Estimating sequencing error models using DADA2's parametric algorithm.
3. **Sample Inference (Denoising)**: Applying the core DADA2 algorithm to resolve exact Amplicon Sequence Variants (ASVs).
4. **Chimera Removal & ASV Table Construction**: Removing chimeric sequences and generating the final abundance table.

In [7]:
import os
import subprocess

# Define absolute paths
project_dir = "/home/azureuser/Microbiome_project"
trimmed_dir = os.path.join(project_dir, "trimmed_data")
container_path = os.path.join(project_dir, "dada2.sif")

# Verify directory exists via Python
if os.path.exists(trimmed_dir):
  files = [f for f in os.listdir(trimmed_dir) if f.endswith(".fastq")]
  print(f"Success! Found {len(files)} .fastq files in trimmed_data.")
  print("Sample files:", files[:5])
else:
  print("Directory still not found. Check path!")

# Now pass the absolute path to DADA2 inside Apptainer
r_code = f"""
library(dada2)
path <- "{trimmed_dir}"
fnFs <- sort(list.files(path, pattern = "\\\\.fastq$", full.names = TRUE))
sample.names <- sub("\\\\.fastq$", "", basename(fnFs))
cat("Total samples detected by DADA2:", length(fnFs), "\\n")
if(length(fnFs) > 0) {{
  cat("First 5 sample names:\\n")
  print(head(sample.names, 5))
}}
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print("\n--- Apptainer R Output ---")
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)

Success! Found 558 .fastq files in trimmed_data.
Sample files: ['ERR3887894.fastq', 'ERR3888291.fastq', 'ERR3888279.fastq', 'ERR3888263.fastq', 'ERR3888265.fastq']

--- Apptainer R Output ---

R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> library(dada2)
> path <- "/home/azureuser/Microbiome_project/trimmed_data"
> fnFs <- sort(list.files(path, pattern = "\\.f

### Step 2: Quality Profiling of Single-End Reads
Before performing filtering and trimming, we inspect the base-wise quality scores of the sequencing reads. This diagnostic step helps us determine the appropriate truncation length (`truncLen`) where the quality score drops, ensuring we remove low-quality bases while retaining sufficient sequence length for downstream analysis.

In [9]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
trimmed_dir = os.path.join(project_dir, "trimmed_data")
container_path = os.path.join(project_dir, "dada2.sif")

# R code to generate and save quality plots for the first few samples
r_code = f"""
library(dada2)
path <- "{trimmed_dir}"
fnFs <- sort(list.files(path, pattern = "\\\\.fastq$", full.names = TRUE))

# Generate quality profile plots for the first 4 samples and save as PDF
pdf("quality_plots_preview.pdf")
print(plotQualityProfile(fnFs[1:4]))
dev.off()

cat("Quality plots successfully saved to quality_plots_preview.pdf\\n")
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)


R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> library(dada2)
> path <- "/home/azureuser/Microbiome_project/trimmed_data"
> fnFs <- sort(list.files(path, pattern = "\\.fastq$", full.names = TRUE))
> 
> # Generate quality profile plots for the first 4 samples and save as PDF
> pdf("quality_plots_preview.pdf")
> print(plotQualityProfile(fnFs[1:4]))
> dev.off()


### Step 3: Filtering and Trimming (filterAndTrim)
Based on the quality profile inspection, the quality scores remain remarkably high (above 30-35) up to ~400 cycles. We will set `truncLen = 400` to trim off the tail ends where quality drops, while keeping `maxEE = 2` and `maxN = 0` to filter out low-quality reads.

In [10]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
trimmed_dir = os.path.join(project_dir, "trimmed_data")
container_path = os.path.join(project_dir, "dada2.sif")

# R code for filterAndTrim with truncLen = 400
r_code = f"""
library(dada2)
path <- "{trimmed_dir}"
fnFs <- sort(list.files(path, pattern = "\\\\.fastq$", full.names = TRUE))
sample.names <- sub("\\\\.fastq$", "", basename(fnFs))

# Define output directory for filtered files
filt_path <- file.path(path, "filtered")
if (!dir.exists(filt_path)) dir.create(filt_path)
filtFs <- file.path(filt_path, paste0(sample.names, "_filt.fastq.gz"))

# Execute filtering and trimming for all 558 samples
out <- filterAndTrim(
  fwd = fnFs, 
  filt = filtFs, 
  truncLen = 400,      # Set based on quality profile inspection
  maxN = 0,            # No ambiguous bases allowed
  maxEE = 2,           # Maximum expected errors allowed
  truncQ = 2,          # Truncate at quality score < 2
  rm.phix = TRUE,      # Remove PhiX contamination
  compress = TRUE, 
  multithread = TRUE   # Enable multithreading inside the container
)

# Display dimensions and summary of filtering output
print(dim(out))
head(out)
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)


R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> library(dada2)
> path <- "/home/azureuser/Microbiome_project/trimmed_data"
> fnFs <- sort(list.files(path, pattern = "\\.fastq$", full.names = TRUE))
> sample.names <- sub("\\.fastq$", "", basename(fnFs))
> 
> # Define output directory for filtered files
> filt_path <- file.path(path, "filtered")
> if (!dir.exist

### Step 4: Learning Error Rates (learnErrors)
DADA2 learns its error model from the data itself using a parametric error estimation. By pooling a sufficient subset of reads, the `learnErrors()` function estimates the error rates for each possible nucleotide transition, which is critical for accurate ASV inference.

In [11]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
trimmed_dir = os.path.join(project_dir, "trimmed_data")
container_path = os.path.join(project_dir, "dada2.sif")

# R code for learnErrors
r_code = f"""
library(dada2)
path <- "{trimmed_dir}"
filt_path <- file.path(path, "filtered")

# Get filtered filenames
filtFs <- sort(list.files(filt_path, pattern = "_filt.fastq.gz$", full.names = TRUE))

# Learn error rates using multithreading
cat("Starting error learning...\n")
errF <- learnErrors(filtFs, multithread = TRUE)

# Plot error rates to inspect convergence
pdf("error_rates_plot.pdf")
print(plotErrors(errF, nominalQ = TRUE))
dev.off()

cat("Error rates successfully learned and plot saved to error_rates_plot.pdf\n")
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)


R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> library(dada2)
> path <- "/home/azureuser/Microbiome_project/trimmed_data"
> filt_path <- file.path(path, "filtered")
> 
> # Get filtered filenames
> filtFs <- sort(list.files(filt_path, pattern = "_filt.fastq.gz$", full.names = TRUE))
> 
> # Learn error rates using multithreading
> cat("Starting error learning..

### Step 5: Sample Inference (dada)
With the error rates successfully learned, we now apply the core sequence-variant inference algorithm (`dada()`) to the filtered reads. This step removes sequencing errors and resolves exact biological variants (ASVs) for each sample.

In [13]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
trimmed_dir = os.path.join(project_dir, "trimmed_data")
container_path = os.path.join(project_dir, "dada2.sif")

# Combined R code: Learn/Ensure error rates and run sample inference immediately
r_code = f"""
library(dada2)
path <- "{trimmed_dir}"
filt_path <- file.path(path, "filtered")

# Get filtered filenames and sample names
filtFs <- sort(list.files(filt_path, pattern = "_filt.fastq.gz$", full.names = TRUE))
sample.names <- sub("_filt.fastq.gz$", "", basename(filtFs))
names(filtFs) <- sample.names

# 1. Re-learn or ensure errF is present (learnErrors takes just a few seconds on pre-filtered reads)
cat("Learning/Confirming error rates...\n")
errF <- learnErrors(filtFs, multithread = TRUE)

# 2. Perform sample inference using dada()
cat("Starting sample inference (dada) across all 558 samples...\n")
dadaFs <- dada(filtFs, err = errF, multithread = TRUE)

# Inspect the dada-class object for the first sample
cat("Sample inference completed successfully!\n")
print(dadaFs[[1]])
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)


R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> library(dada2)
> path <- "/home/azureuser/Microbiome_project/trimmed_data"
> filt_path <- file.path(path, "filtered")
> 
> # Get filtered filenames and sample names
> filtFs <- sort(list.files(filt_path, pattern = "_filt.fastq.gz$", full.names = TRUE))
> sample.names <- sub("_filt.fastq.gz$", "", basename(filtFs)

In [2]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
trimmed_dir = os.path.join(project_dir, "trimmed_data")
container_path = os.path.join(project_dir, "dada2.sif")

r_code = f"""
library(dada2)
path <- "{trimmed_dir}"
filt_path <- file.path(path, "filtered")
filtFs <- sort(list.files(filt_path, pattern = "_filt.fastq.gz$", full.names = TRUE))
sample.names <- sub("_filt.fastq.gz$", "", basename(filtFs))
names(filtFs) <- sample.names

# 1. Learn errors and run dada inference
cat("Learning error rates...\n")
errF <- learnErrors(filtFs, multithread = TRUE)

cat("Running sample inference (dada)...\n")
dadaFs <- dada(filtFs, err = errF, multithread = TRUE)

# 2. Construct sequence table
cat("Constructing sequence table...\n")
seqtab <- makeSequenceTable(dadaFs)
cat("Dimensions of raw sequence table (samples x ASVs):\n")
print(dim(seqtab))

# 3. Remove chimeras
cat("Removing chimeras...\n")
seqtab.nochim <- removeBimeraDenovo(seqtab, method = "consensus", multithread = TRUE, verbose = TRUE)

cat("Dimensions of chimera-free sequence table:\n")
print(dim(seqtab.nochim))

# Percentage of non-chimeric reads
cat("Percentage of non-chimeric reads:\n")
print(sum(seqtab.nochim) / sum(seqtab))

# Save final table
saveRDS(seqtab.nochim, "seqtab_final.rds")
cat("Final ASV table successfully saved to seqtab_final.rds!\n")
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)


R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> library(dada2)
> path <- "/home/azureuser/Microbiome_project/trimmed_data"
> filt_path <- file.path(path, "filtered")
> filtFs <- sort(list.files(filt_path, pattern = "_filt.fastq.gz$", full.names = TRUE))
> sample.names <- sub("_filt.fastq.gz$", "", basename(filtFs))
> names(filtFs) <- sample.names
> 
> # 1. Lea

### Conclusion: Denoising, ASV Table & Chimera Removal
In this notebook, we successfully completed the core DADA2 sequence variant inference pipeline:
* **Sample Inference (`dada`)**: Processed 558 samples to resolve exact amplicon sequence variants (ASVs).
* **Sequence Table Construction**: Combined all samples into a raw ASV table containing 19,790 features.
* **Chimera Removal (`removeBimeraDenovo`)**: Filtered out chimeric sequences, resulting in a final high-quality table of **6,013 clean ASVs** across all 558 samples, with an excellent non-chimeric read retention rate of **94.05%**.
* **Output**: The final clean table was successfully saved as `seqtab_final.rds`, ready for taxonomic assignment and downstream ecological analyses.